# Pertemuan 3 — Data Cleaning: Missing Values, Outlier & Ekstraksi Data

**Mata Kuliah:** Pengantar Data Science (Kode: 200302305)  
**Program Studi:** PJJ Informatika  
**Semester:** 4  
**Dosen:** Syahid Abdullah, S.Si, M.Kom

---

**Nama :** Junior Dany Wibisono  
**NIM  :** 250401020098  
**Kelas :** IF401  
**Angkatan:** 2025  
**Tanggal:** 10 Mei 2026

---

## Tujuan Notebook
Notebook ini merupakan aktivitas hands-on Pertemuan 3 untuk:
1. Mendeteksi dan menangani missing values dengan strategi yang tepat
2. Menghapus data duplikat dan menormalisasi inkonsistensi string
3. Mendeteksi dan menangani outlier menggunakan metode IQR Fence
4. Mengakses data dari REST API publik (JSONPlaceholder)
5. Mengekspor dataset yang telah bersih ke file CSV

## LANGKAH 0 — Import Library & Muat Dataset

Mengimpor seluruh library yang diperlukan dan memuat dataset `housing_dirty.csv`.

Dataset ini berisi data properti perumahan dengan berbagai masalah kualitas data yang sengaja ditambahkan:
- Missing values pada kolom numerik dan kategorik
- Baris duplikat
- Outlier pada harga dan tahun bangun
- Inkonsistensi format string

In [1]:
# LANGKAH 0: Import library
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimpor!')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

Library berhasil diimpor!
Pandas  : 2.2.2
NumPy   : 2.0.2


In [2]:
# Muat dataset dari Google Drive (link dari Modul Pertemuan 3)
URL_GDRIVE = 'https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgnstULfIRo'

try:
    df = pd.read_csv(URL_GDRIVE)
    print(f'Dataset berhasil dimuat dari Google Drive!')
except Exception:
    # Fallback: buat dataset simulasi jika Google Drive tidak dapat diakses
    print('Gagal memuat dari GDrive, membuat dataset simulasi...')
    np.random.seed(42)
    n = 130
    kota_list  = ['jakarta', 'Jakarta', 'JAKARTA', 'bandung', 'Bandung', 'surabaya', 'Surabaya']
    kondisi_list = ['Baik', 'baik', 'BAIK', 'sedang', 'Sedang', 'buruk', 'Buruk']
    data = {
        'id'           : range(1, n + 1),
        'luas_m2'      : np.where(np.random.rand(n) < 0.10, np.nan,
                                  np.random.uniform(36, 250, n).round(1)),
        'harga_juta'   : np.where(np.random.rand(n) < 0.08, np.nan,
                                  np.random.uniform(200, 5000, n).round(1)),
        'kota'         : np.random.choice(kota_list, n),
        'kamar'        : np.where(np.random.rand(n) < 0.07, np.nan,
                                  np.random.choice([2, 3, 4, 5], n)),
        'tahun_bangun' : np.random.choice(
                            list(range(1990, 2024)) + [1900, 1800, 9999], n),
        'kondisi'      : np.random.choice(kondisi_list, n),
    }
    df = pd.DataFrame(data)
    # Tambahkan outlier harga
    df.loc[5, 'harga_juta']  = 999999
    df.loc[12, 'harga_juta'] = -500
    # Tambahkan duplikat
    df = pd.concat([df, df.iloc[[0, 1, 2]]], ignore_index=True)
    print('Dataset simulasi berhasil dibuat!')

print(f'\nShape awal  : {df.shape}')
print(f'Kolom       : {df.columns.tolist()}')

Gagal memuat dari GDrive, membuat dataset simulasi...
Dataset simulasi berhasil dibuat!

Shape awal  : (133, 7)
Kolom       : ['id', 'luas_m2', 'harga_juta', 'kota', 'kamar', 'tahun_bangun', 'kondisi']


## LANGKAH 1 — Eksplorasi Awal

Sebelum melakukan cleaning, kita perlu memahami kondisi dataset terlebih dahulu melalui:
- Ukuran dan tipe data (`shape`, `dtypes`, `info()`)
- Ringkasan statistik (`describe()`)
- Jumlah dan proporsi missing values (`isnull().sum()`)

In [3]:
# LANGKAH 1a: Info dasar
print('=== INFORMASI DATASET ===')
print(f'Jumlah baris  : {df.shape[0]}')
print(f'Jumlah kolom  : {df.shape[1]}')
print()
df.info()

=== INFORMASI DATASET ===
Jumlah baris  : 133
Jumlah kolom  : 7

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            133 non-null    int64  
 1   luas_m2       117 non-null    float64
 2   harga_juta    117 non-null    float64
 3   kota          133 non-null    object 
 4   kamar         123 non-null    float64
 5   tahun_bangun  133 non-null    int64  
 6   kondisi       133 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.4+ KB


In [4]:
# LANGKAH 1b: Lima baris pertama
print('=== LIMA BARIS PERTAMA ===')
df.head(10)

=== LIMA BARIS PERTAMA ===


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,125.3,4954.4,Bandung,3.0,1995,Sedang
1,2,83.5,2180.6,Jakarta,5.0,1997,BAIK
2,3,61.7,1985.7,Surabaya,4.0,2012,Sedang
3,4,108.2,3926.8,jakarta,4.0,2015,BAIK
4,5,237.8,1835.9,surabaya,2.0,2001,sedang
5,6,105.2,999999.0,JAKARTA,5.0,2015,Sedang
6,7,NaN,4320.4,JAKARTA,5.0,2002,sedang
7,8,186.4,2259.2,jakarta,3.0,2007,baik
8,9,113.8,3804.2,Bandung,5.0,2014,baik
9,10,244.0,3821.8,jakarta,5.0,2022,Baik


In [5]:
# LANGKAH 1c: Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe().round(2)

=== STATISTIK DESKRIPTIF ===


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,133.00,117.00,117.00,123.00,133.00
mean,64.07,139.78,11224.25,3.52,2243.03
std,38.43,63.35,92213.21,1.12,1371.24
min,1.00,37.10,-500.00,2.00,1800.00
25%,31.00,87.80,1548.60,2.50,1997.00
50%,64.00,140.70,2733.00,4.00,2009.00
75%,97.00,186.40,4144.90,4.00,2017.00
max,130.00,246.90,999999.00,5.00,9999.00


In [6]:
# LANGKAH 1d: Audit missing values
print('=== AUDIT MISSING VALUES ===')
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

audit_missing = pd.DataFrame({
    'Jumlah Missing' : missing_count,
    'Persentase (%)'  : missing_pct
})
print(audit_missing)
print(f'\nTotal sel yang missing: {df.isnull().sum().sum()}')

=== AUDIT MISSING VALUES ===
              Jumlah Missing  Persentase (%)
id                         0            0.00
luas_m2                   16           12.03
harga_juta                16           12.03
kota                       0            0.00
kamar                     10            7.52
tahun_bangun               0            0.00
kondisi                    0            0.00

Total sel yang missing: 42


In [7]:
# LANGKAH 1e: Cek duplikat dan nilai unik kolom kategorik
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')
print()
print('Nilai unik kolom KOTA:')
print(df['kota'].value_counts())
print()
print('Nilai unik kolom KONDISI:')
print(df['kondisi'].value_counts())

Jumlah baris duplikat: 3

Nilai unik kolom KOTA:
kota
Bandung     21
Jakarta     21
Surabaya    21
jakarta     20
surabaya    20
JAKARTA     19
bandung     11
Name: count, dtype: int64

Nilai unik kolom KONDISI:
kondisi
Sedang    25
baik      24
buruk     20
BAIK      19
sedang    16
Baik      16
Buruk     13
Name: count, dtype: int64


## LANGKAH 2 — Hapus Duplikat

**Data duplikat** adalah baris yang merepresentasikan observasi yang sama lebih dari sekali. Duplikat mendistorsi statistik (mean, count) dan menyebabkan model machine learning overfitting.

Strategi:
- Identifikasi semua baris yang duplikat dengan `duplicated(keep=False)`
- Hapus duplikat, pertahankan kemunculan pertama dengan `drop_duplicates(keep='first')`

In [8]:
# LANGKAH 2: Hapus data duplikat
print(f'Shape sebelum hapus duplikat: {df.shape}')
print(f'Jumlah baris duplikat       : {df.duplicated().sum()}')

# Tampilkan baris yang duplikat
df_dup = df[df.duplicated(keep=False)]
if len(df_dup) > 0:
    print(f'\nBaris yang duplikat (semua kemunculan):')
    print(df_dup)

# Hapus duplikat
df.drop_duplicates(keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nShape setelah hapus duplikat: {df.shape}')
print(f'Sisa baris duplikat         : {df.duplicated().sum()}')

Shape sebelum hapus duplikat: (133, 7)
Jumlah baris duplikat       : 3

Baris yang duplikat (semua kemunculan):
     id  luas_m2  harga_juta      kota  kamar  tahun_bangun kondisi
0     1    125.3      4954.4   Bandung    3.0          1995  Sedang
1     2     83.5      2180.6   Jakarta    5.0          1997    BAIK
2     3     61.7      1985.7  Surabaya    4.0          2012  Sedang
130   1    125.3      4954.4   Bandung    3.0          1995  Sedang
131   2     83.5      2180.6   Jakarta    5.0          1997    BAIK
132   3     61.7      1985.7  Surabaya    4.0          2012  Sedang

Shape setelah hapus duplikat: (130, 7)
Sisa baris duplikat         : 0


## LANGKAH 3 — Normalisasi String

**Inkonsistensi format** seperti 'jakarta', 'Jakarta', 'JAKARTA' menyebabkan `groupby` menghitung kategori yang sama sebagai kelompok yang berbeda.

Strategi:
- `str.strip()` — hapus spasi di awal dan akhir
- `str.title()` — jadikan huruf kapital di awal setiap kata (untuk nama kota)
- `str.lower()` — jadikan semua huruf kecil (untuk kondisi)

In [9]:
# LANGKAH 3: Normalisasi string
print('=== SEBELUM NORMALISASI ===')
print('Nilai unik KOTA   :', df['kota'].unique())
print('Nilai unik KONDISI:', df['kondisi'].unique())

# Normalisasi kolom kota → Title Case
df['kota'] = df['kota'].str.strip().str.title()

# Normalisasi kolom kondisi → lower case
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\n=== SETELAH NORMALISASI ===')
print('Nilai unik KOTA   :', df['kota'].unique())
print('Nilai unik KONDISI:', df['kondisi'].unique())

print('\nDistribusi KOTA:')
print(df['kota'].value_counts())

=== SEBELUM NORMALISASI ===
Nilai unik KOTA   : ['Bandung' 'Jakarta' 'Surabaya' 'jakarta' 'surabaya' 'JAKARTA' 'bandung']
Nilai unik KONDISI: ['Sedang' 'BAIK' 'sedang' 'baik' 'Baik' 'Buruk' 'buruk']

=== SETELAH NORMALISASI ===
Nilai unik KOTA   : ['Bandung' 'Jakarta' 'Surabaya']
Nilai unik KONDISI: ['sedang' 'baik' 'buruk']

Distribusi KOTA:
kota
Jakarta     59
Surabaya    40
Bandung     31
Name: count, dtype: int64


## LANGKAH 4 — Imputasi Missing Values

**Strategi imputasi yang dipilih:**
- `luas_m2` → imputasi **median** (robust terhadap outlier, distribusi mungkin skewed)
- `harga_juta` → imputasi **median** (harga properti biasanya right-skewed)
- `kamar` → imputasi **modus** (data diskrit, nilai yang paling sering muncul lebih logis)

In [10]:
# LANGKAH 4: Imputasi missing values
print('=== MISSING SEBELUM IMPUTASI ===')
print(df[['luas_m2', 'harga_juta', 'kamar']].isnull().sum())

# Imputasi median untuk kolom numerik
median_luas   = df['luas_m2'].median()
median_harga  = df['harga_juta'].median()
modus_kamar   = df['kamar'].mode()[0]

print(f'\nNilai imputasi:')
print(f'  Median luas_m2    : {median_luas:.1f} m²')
print(f'  Median harga_juta : Rp {median_harga:.1f} juta')
print(f'  Modus kamar       : {modus_kamar} kamar')

# Terapkan imputasi
df['luas_m2']    = df['luas_m2'].fillna(median_luas)
df['harga_juta'] = df['harga_juta'].fillna(median_harga)
df['kamar']      = df['kamar'].fillna(modus_kamar)

print('\n=== MISSING SETELAH IMPUTASI ===')
print(df[['luas_m2', 'harga_juta', 'kamar']].isnull().sum())
print(f'\nTotal missing yang tersisa: {df.isnull().sum().sum()}')

=== MISSING SEBELUM IMPUTASI ===
luas_m2       16
harga_juta    16
kamar         10
dtype: int64

Nilai imputasi:
  Median luas_m2    : 141.9 m²
  Median harga_juta : Rp 2785.3 juta
  Modus kamar       : 4.0 kamar

=== MISSING SETELAH IMPUTASI ===
luas_m2       0
harga_juta    0
kamar         0
dtype: int64

Total missing yang tersisa: 0


## LANGKAH 5 — Deteksi dan Penanganan Outlier (IQR Fence)

**Metode IQR Fence** mendefinisikan batas outlier sebagai:
- **Batas bawah** = Q1 − 1.5 × IQR
- **Batas atas**  = Q3 + 1.5 × IQR

Nilai di luar batas tersebut dianggap outlier. Strategi penanganan yang digunakan adalah **capping (clip)** — nilai outlier dipotong ke batas, bukan dihapus — agar jumlah baris tetap terjaga.

Kolom yang diproses: `harga_juta`, `luas_m2`, `tahun_bangun`

In [11]:
# LANGKAH 5: Deteksi dan tangani outlier menggunakan IQR Fence + capping

def iqr_fence_info(series, nama_kolom):
    """Menampilkan informasi IQR Fence dan jumlah outlier."""
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f'\n--- {nama_kolom} ---')
    print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Batas bawah : {lower:.2f}')
    print(f'  Batas atas  : {upper:.2f}')
    print(f'  Outlier     : {len(outliers)} baris')
    return lower, upper

print('=== DETEKSI OUTLIER (SEBELUM CAPPING) ===')
target_cols = ['harga_juta', 'luas_m2', 'tahun_bangun']
batas = {}
for col in target_cols:
    lower, upper = iqr_fence_info(df[col], col)
    batas[col] = (lower, upper)

print('\n=== TERAPKAN CAPPING ===')
for col in target_cols:
    lo, hi = batas[col]
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lower=lo, upper=hi)
    after_min,  after_max  = df[col].min(), df[col].max()
    print(f'{col}: [{before_min:.1f}, {before_max:.1f}] → [{after_min:.1f}, {after_max:.1f}]')

=== DETEKSI OUTLIER (SEBELUM CAPPING) ===

--- harga_juta ---
  Q1=1697.45, Q3=3929.43, IQR=2231.98
  Batas bawah : -1650.51
  Batas atas  : 7277.39
  Outlier     : 1 baris

--- luas_m2 ---
  Q1=93.50, Q3=183.18, IQR=89.68
  Batas bawah : -41.01
  Batas atas  : 317.69
  Outlier     : 0 baris

--- tahun_bangun ---
  Q1=1997.00, Q3=2017.00, IQR=20.00
  Batas bawah : 1967.00
  Batas atas  : 2047.00
  Outlier     : 8 baris

=== TERAPKAN CAPPING ===
harga_juta: [-500.0, 999999.0] → [-500.0, 7277.4]
luas_m2: [37.1, 246.9] → [37.1, 246.9]
tahun_bangun: [1800.0, 9999.0] → [1967.0, 2047.0]


In [12]:
# Cek outlier setelah capping
print('=== VERIFIKASI OUTLIER SETELAH CAPPING ===')
for col in target_cols:
    lo, hi = batas[col]
    sisa = df[(df[col] < lo) | (df[col] > hi)]
    print(f'{col}: sisa outlier = {len(sisa)} baris')

=== VERIFIKASI OUTLIER SETELAH CAPPING ===
harga_juta: sisa outlier = 0 baris
luas_m2: sisa outlier = 0 baris
tahun_bangun: sisa outlier = 0 baris


## LANGKAH 6 — Validasi Akhir & Ekspor

Sebelum menyimpan dataset, lakukan validasi:
- Tidak ada missing values tersisa
- Tidak ada duplikat tersisa
- Rentang nilai numerik masuk akal

In [13]:
# LANGKAH 6: Validasi akhir
print('=== VALIDASI AKHIR ===')

total_missing  = df.isnull().sum().sum()
total_duplikat = df.duplicated().sum()

assert total_missing  == 0, f'GAGAL: Masih ada {total_missing} missing values!'
assert total_duplikat == 0, f'GAGAL: Masih ada {total_duplikat} duplikat!'

print(f'✓ Total missing values : {total_missing}')
print(f'✓ Total duplikat       : {total_duplikat}')
print(f'✓ Shape akhir dataset  : {df.shape}')
print('\n=== STATISTIK AKHIR ===')
df.describe().round(2)

=== VALIDASI AKHIR ===
✓ Total missing values : 0
✓ Total duplikat       : 0
✓ Shape akhir dataset  : (130, 7)

=== STATISTIK AKHIR ===


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.00,130.00,130.00,130.00,130.00
mean,65.50,141.19,2738.16,3.55,2007.53
std,37.67,59.45,1502.76,1.09,14.03
min,1.00,37.10,-500.00,2.00,1967.00
25%,33.25,93.50,1697.45,3.00,1997.00
50%,65.50,141.90,2785.35,4.00,2009.50
75%,97.75,183.18,3929.42,4.00,2017.00
max,130.00,246.90,7277.39,5.00,2047.00


In [14]:
# Ekspor dataset bersih
NAMA_FILE = 'housing_clean.csv'
df.to_csv(NAMA_FILE, index=False)
print(f'Dataset bersih berhasil disimpan sebagai "{NAMA_FILE}"')
print(f'Ukuran file : {df.shape[0]} baris × {df.shape[1]} kolom')

Dataset bersih berhasil disimpan sebagai "housing_clean.csv"
Ukuran file : 130 baris × 7 kolom


## LANGKAH 7 — Ekstraksi Data dari REST API (JSONPlaceholder)

**REST API (Representational State Transfer)** adalah arsitektur paling umum untuk menyediakan data melalui web. Kita akan berlatih mengakses dua endpoint dari JSONPlaceholder — API gratis tanpa autentikasi:
- `/users` — daftar pengguna
- `/posts?userId=1` — daftar postingan dari user tertentu

Library `requests` digunakan untuk mengirim HTTP GET request dan menerima respons JSON.

In [15]:
# LANGKAH 7a: Ambil data dari endpoint /users
BASE_URL = 'https://jsonplaceholder.typicode.com'

print('Mengirim GET request ke /users ...')
response = requests.get(f'{BASE_URL}/users', timeout=10)

print(f'Status Code: {response.status_code}')

if response.status_code == 200:
    data_users = response.json()
    df_users = json_normalize(data_users, sep='_')
    print(f'Berhasil! {len(df_users)} pengguna diterima.')
    print(f'Kolom: {df_users.columns.tolist()}')
else:
    print(f'Error: {response.status_code}')

Mengirim GET request ke /users ...
Status Code: 200
Berhasil! 10 pengguna diterima.
Kolom: ['id', 'name', 'username', 'email', 'phone', 'website', 'address_street', 'address_suite', 'address_city', 'address_zipcode', 'address_geo_lat', 'address_geo_lng', 'company_name', 'company_catchPhrase', 'company_bs']


In [16]:
# Tampilkan subset kolom yang relevan
kolom_tampil = ['id', 'name', 'username', 'email', 'address_city', 'company_name']
print('=== DATA PENGGUNA DARI API ===')
df_users[kolom_tampil]

=== DATA PENGGUNA DARI API ===


,id,name,username,email,address_city,company_name
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,South Christy,Considine-Lockman
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,Howemouth,Johns Group
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,Aliyaview,Abernathy Group
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Bartholomebury,Yost and Sons
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,Lebsackbury,Hoeger LLC


In [17]:
# LANGKAH 7b: Ambil postingan dari user tertentu (query parameter)
USER_ID = 1
params  = {'userId': USER_ID}

resp_posts = requests.get(f'{BASE_URL}/posts', params=params, timeout=10)
print(f'Status Code: {resp_posts.status_code}')

if resp_posts.status_code == 200:
    df_posts = pd.DataFrame(resp_posts.json())
    print(f'Postingan dari user {USER_ID}: {len(df_posts)} post')
    print(df_posts[['id', 'userId', 'title']].head())
else:
    print(f'Error: {resp_posts.status_code}')

Status Code: 200
Postingan dari user 1: 10 post
   id  userId                                              title
0   1       1  sunt aut facere repellat provident occaecati e...
1   2       1                                       qui est esse
2   3       1  ea molestias quasi exercitationem repellat qui...
3   4       1                               eum et est occaecati
4   5       1                                 nesciunt quas odio


In [18]:
# Eksplorasi dasar DataFrame dari API
print('=== STATISTIK DESKRIPTIF DATA PENGGUNA ===')
print(f'Total pengguna  : {len(df_users)}')
print(f'Total kolom     : {len(df_users.columns)}')
print(f'Missing values  : {df_users.isnull().sum().sum()}')
print()
print('Distribusi berdasarkan kota:')
print(df_users['address_city'].value_counts())

=== STATISTIK DESKRIPTIF DATA PENGGUNA ===
Total pengguna  : 10
Total kolom     : 15
Missing values  : 0

Distribusi berdasarkan kota:
address_city
Gwenborough       1
Wisokyburgh       1
McKenziehaven     1
South Elvis       1
Roscoeview        1
South Christy     1
Howemouth         1
Aliyaview         1
Bartholomebury    1
Lebsackbury       1
Name: count, dtype: int64


## Ringkasan Pipeline Cleaning

| Langkah | Tindakan | Hasil |
|---------|----------|-------|
| 0 | Load dataset | Shape awal berhasil dimuat |
| 1 | Eksplorasi awal | Ditemukan missing values, duplikat, outlier, inkonsistensi |
| 2 | Hapus duplikat | `drop_duplicates(keep='first')` |
| 3 | Normalisasi string | `str.strip().str.title()` untuk kota, `str.lower()` untuk kondisi |
| 4 | Imputasi missing | Median untuk numerik, modus untuk kamar |
| 5 | Tangani outlier | IQR Fence + capping (`clip`) |
| 6 | Validasi & ekspor | `assert` zero missing, zero duplikat → `housing_clean.csv` |
| 7 | Akses REST API | GET JSONPlaceholder `/users` dan `/posts` |

## Refleksi Singkat

Pertemuan 3 mengajarkan bahwa Data Scientist menghabiskan sebagian besar waktunya pada tahap persiapan data. Prinsip **'Garbage In, Garbage Out'** menegaskan bahwa kualitas model machine learning sangat bergantung pada kualitas data masukan. Melalui pipeline cleaning ini, saya memahami pentingnya setiap langkah: dari deteksi duplikat, imputasi missing values yang tepat sasaran, hingga penanganan outlier dengan metode IQR Fence. Kemampuan mengakses REST API juga memperluas sumber data yang dapat digunakan dalam proyek Data Science nyata.

---
**Keterbatasan & Pertanyaan:** Dataset yang digunakan adalah simulasi, sehingga hasil cleaning mungkin tidak mencerminkan kompleksitas data nyata. Pertanyaan yang muncul: kapan sebaiknya menggunakan imputasi KNN dibanding imputasi median? Bagaimana menentukan batas IQR yang optimal untuk domain data yang berbeda-beda?